# Learning which VRP solver works best

Compact fast-mode notebook for the frozen final experiment. It loads saved artifacts instead of rerunning the expensive solver experiment. Methodology follows Asín-Achá et al. (2024), *Networks* 84(4):465-480.

## 1. Scope

Symmetric Euclidean CVRP, one depot, homogeneous capacity, positive heterogeneous demands, routes returning to the depot.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

BASE = Path('reports/final_ml_experiment')
assert BASE.exists(), BASE
summary = json.loads((BASE / 'ml_summary.json').read_text())
metrics = pd.read_csv(BASE / 'model_metrics.csv')
experiment = pd.read_csv(BASE / 'experiment_table.csv')
features = json.loads((BASE / 'feature_columns.json').read_text())
summary['n_rows'], summary['train_rows'], summary['test_rows']


## 2. Problem Introduction

The Capacitated Vehicle Routing Problem (CVRP): a depot hosts an unlimited fleet of identical vehicles of capacity $Q$; every customer $i$ has demand $q_i > 0$ and must be visited exactly once; each route starts and ends at the depot and its total demand may not exceed $Q$; the objective is to minimize total travelled distance. CVRP is NP-hard, which motivates both exact reference solutions on small instances and fast heuristics at scale.

**MIP formulation used here (gurobipy)** — two-index arc formulation with load propagation:

$$\min \sum_{i \ne j} c_{ij} x_{ij}$$

subject to: each customer is entered and left exactly once; depot departures equal returns; and load constraints $u_j \ge u_i + q_j - Q(1 - x_{ij})$ with $q_i \le u_i \le Q$, which eliminate subtours and enforce capacity simultaneously.

Below, the tiny manual instance and two self-created synthetic instances are solved with gurobipy to proven optimality and with all five portfolio heuristics; each heuristic's gap to the optimum is shown.

In [ ]:
exact = pd.read_csv('reports/exact_baseline/exact_vs_heuristic.csv')
exact[['instance','method','cost','runtime_seconds','optimal','gap_to_exact_percent']]

## 3. Benchmark Instance E-n51-k5

The CVRPLIB benchmark E-n51-k5 (proven optimum 521) is solved heuristically; the exact model exceeds the free size-limited Gurobi license (51 nodes → 2550 binary arc variables > 2000), which is recorded honestly. The trained Random Forest selector predicts the best portfolio solver for this out-of-distribution instance.

In [ ]:
demo = json.loads(Path('reports/exact_baseline/benchmark_demo.json').read_text())
print('best known:', demo['best_known_cost'])
print('exact attempt:', demo['exact_attempt']['status'], '-', demo['exact_attempt']['failure_reason'])
print('selector predicted:', demo['selector']['predicted_solver'],
      '| best in portfolio:', demo['selector']['prediction_is_best_in_portfolio'])
pd.DataFrame(demo['heuristics'])[['method','cost','gap_to_best_known_percent','runtime_seconds']]

## 4. Fixed Time Budget

Every portfolio solver runs under the same fixed wall-clock budget (`TIME_BUDGET_SECONDS = 20.0`); a solver that does not return a feasible solution in time is recorded as failed. The label `best_solver` therefore means “best solution within the fixed time budget”. In the frozen experiment every feasible run finished within the budget (max 16.98 s), so the frozen labels coincide exactly with best-within-20s labels.

In [ ]:
from cvrp_algorithm_selection.solvers import TIME_BUDGET_SECONDS
runtime_cols = [c for c in experiment.columns if c.endswith('_runtime')]
print('budget:', TIME_BUDGET_SECONDS, 'seconds')
experiment[runtime_cols].max().round(3)

## 5. Final Portfolio

The main selector uses five VeRyPy-based heuristics. Gaskell remains implemented/audited but is excluded from ML because it won too few development instances.

In [ ]:
summary['label_counts']


## 6. Distance Convention

The final experiment uses `cvrplib_euc_2d`, the rounded CVRPLIB/TSPLIB convention. Costs and labels are not mixed with floating Euclidean values.

In [ ]:
experiment['distance_convention'].value_counts()


## 7. Dataset Design

The final synthetic dataset is a factorial-style grid over spatial family, depot position, demand regime, customer count, and seed.

In [ ]:
experiment[['spatial_family','depot_position','demand_regime','customer_count','seed']].nunique()


## 8. Feature Matrix

Model features are pre-solution instance features only. IDs, seeds, generator labels, solver costs, runtimes, feasibility columns, and labels are excluded.

In [ ]:
len(features), features[:10]


## 9. Leakage Check

This fast check confirms prohibited solver/outcome columns are not in the saved feature manifest.

In [ ]:
prohibited_fragments = ['cost', 'runtime', 'feasible', 'failure', 'best_solver', 'best_cost', 'instance_id', 'seed', 'spatial_family', 'depot_position', 'demand_regime']
bad = [c for c in features if any(p in c for p in prohibited_fragments)]
assert bad == [], bad
'no prohibited feature columns'


## 10. Label Distribution

![Label distribution](reports/final_ml_experiment/plots/label_distribution.png)

In [ ]:
pd.Series(summary['label_counts']).sort_values(ascending=False)


## 11. Baselines And Models

Evaluation includes Dummy classifiers, SBS selected from training data only, VBS oracle, KNN, Decision Tree, Random Forest, and Gradient Boosting.

In [ ]:
metrics[['method','accuracy','balanced_accuracy','macro_f1','mean_regret','mean_cost_ratio','within_1_percent','within_5_percent']]


## 12. Accuracy

![Accuracy and balanced accuracy](reports/final_ml_experiment/plots/accuracy_balanced_accuracy.png)

## 13. Routing Regret

![Mean regret](reports/final_ml_experiment/plots/mean_regret.png)

In [ ]:
metrics[['method','mean_regret','median_regret','p90_regret','p95_regret','max_regret']].sort_values('mean_regret')


## 14. Cost Ratio

![Mean cost ratio](reports/final_ml_experiment/plots/mean_cost_ratio.png)

## 15. Within-Threshold Quality

![Within thresholds](reports/final_ml_experiment/plots/within_thresholds.png)

## 16. Best Model

Random Forest is the best verified ML model by mean regret. It improves mean regret versus SBS while using only pre-solution instance features.

In [ ]:
metrics.loc[metrics['method'].isin(['RandomForest','SBS','VBS']), ['method','accuracy','mean_regret','within_1_percent','within_5_percent','mean_selected_solver_runtime']]


### Normalized selector metric $\hat{m}$ (Asín-Achá et al. 2024)

$\hat{m}_s = (m_s - m_{VBS}) / (m_{SBS} - m_{VBS})$ where $m_s$ is the mean cost ratio to VBS: 0 = selects like the VBS oracle, 1 = no better than always running the SBS.

In [ ]:
sbs_ratio = float(metrics.loc[metrics['method'] == 'SBS', 'mean_cost_ratio'].iloc[0])
mhat = ((metrics.set_index('method')['mean_cost_ratio'] - 1.0) / (sbs_ratio - 1.0)).round(3)
mhat.sort_values()

## 17. Confusion Matrix

![Random Forest confusion matrix](reports/final_ml_experiment/plots/best_model_confusion_matrix.png)

## 18. Feature Importance

![Random Forest feature importance](reports/final_ml_experiment/plots/random_forest_feature_importance.png)

In [ ]:
pd.read_csv(BASE / 'random_forest_feature_importance.csv').head(10)


## 19. Grouped Performance

Grouped metrics are saved for spatial family, depot position, demand regime, customer count, and true best solver.

In [ ]:
grouped = json.loads((BASE / 'grouped_metrics.json').read_text())
pd.DataFrame(grouped['customer_count'])


## 20. Solver Failures

Failures are recorded without aborting the experiment. The one Wren-Holliday run that exceeded the fixed 20 s budget is handled by labelling from the remaining feasible solvers; at deployment the meta-solver falls back to the SBS if the predicted solver fails.

In [ ]:
summary['solver_failure_counts']


## 21. Integrity Checks

These checks recompute the label-minimum rule and confirm VBS metrics.

In [ ]:
from cvrp_algorithm_selection.ml_experiment import final_solver_prefixes, validate_label_minima
validate_label_minima(experiment, final_solver_prefixes())
vbs = metrics[metrics['method'] == 'VBS'].iloc[0]
assert abs(vbs['mean_regret']) < 1e-12
assert abs(vbs['mean_cost_ratio'] - 1.0) < 1e-12
'integrity checks passed'


## 22. Fast Reproduction Command

Run tests with `uv run pytest`. Run this notebook in fast mode by executing its code cells against saved outputs; no solver regeneration is required.

## 23. Conclusion

The portfolio is complementary enough for supervised selection, and Random Forest beats SBS on mean routing regret, closing about 26% of the SBS-to-VBS gap ($\hat{m} = 0.74$). Exact reference solutions from gurobipy confirm the heuristic portfolio stays within a few percent of optimal on small instances. The strongest limitations are synthetic training data, heuristic labels rather than exact optima for the selector, and runtime being bounded by the fixed budget but not rewarded in the label objective.